In [3]:
#
# Copyright 2025 Keysight Technologies Inc.
#
# This file generates the asset "calibration.qcs". Run `python calibration.py`
# to generate the asset.
#

import numpy as np

import keysight.qcs as qcs
from keysight.qcs.channels import (
    Channels,
    ConstantEnvelope,
    GaussianEnvelope,
    PhaseIncrement,
    RFWaveform,
    SineEnvelope,
)
from keysight.qcs.programs import CalibrationSet
from keysight.qcs.quantum import (
    GATES,
    PAULIS,
    ParameterizedGate,
    ParametricGate,
    Qudits,
)
from keysight.qcs.variables import Array, Scalar

####################################################################################
#
# Configuration parameters
#
####################################################################################

# define the qubits and matching virtual channels
n_qubits = 2
n_channels = 2
qubits = Qudits(range(n_qubits))
edge_list = [(0, 1)]

xy_awg = Channels(qubits.labels, "xy_channels", absolute_phase=True)
readout_awg = Channels(qubits.labels, "readout_channels", absolute_phase=True)
dig = Channels(qubits.labels, "acquire_channels")

# define the multiqubits targets
pairs = qubits.make_connectivity(edge_list)
topology = "custom"
n_mq = len(pairs)

# initialize the calibration set
calibration_set = CalibrationSet()

####################################################################################
#
# Define shared variables
#
####################################################################################

# the following variables are shared across different operations

xy_pulse_frequencies = Array(
    "xy_pulse_frequencies", dtype=float, value=[5.0e9] * n_qubits
)
"""
The xy pulse frequencies of the individual qubits.
"""

xy_pulse_durations = Array("xy_pulse_durations", dtype=float, value=[30e-9] * n_qubits)
"""
The duration of the xy pulse for each qubit.
"""

x180_pulse_amplitudes = Array(
    "x180_pulse_amplitudes", dtype=float, value=[0.5] * n_qubits
)
"""
The amplitude of the x180 pulse for each qubit.
"""

readout_frequencies = Array(
    "readout_frequencies", dtype=float, value=[6.0e9] * n_qubits
)
"""
The readout frequencies of the individual qubits.
"""

readout_pulse_duration = Array(
    "readout_pulse_duration", dtype=float, value=[100e-9] * n_qubits
)
"""
The readout pulse duration (as of now: common to all qubits).
"""

readout_pulse_delay = Array("readout_pulse_delay", dtype=float, value=[0.0] * n_qubits)
"""
The readout pulse execution delay (as of now: common to all qubits).
"""

readout_pulse_phases = Array(
    "readout_pulse_phases", dtype=float, value=[0.0] * n_qubits
)
"""
The phase of the readout pulse (as of now: common to all qubits).
"""

readout_pulse_amplitudes = Array(
    "readout_pulse_amplitudes", dtype=float, value=[0.1] * n_qubits
)
"""
The amplitude of the readout pulse (as of now: common to all qubits).
"""

acquisition_duration = Array(
    "acquisition_duration", dtype=float, value=[100e-9] * n_qubits
)
"""
The duration of the readout acquisition (as of now: common to all qubits).
"""

acquisition_delay = Array("acquisition_delay", dtype=float, value=[10e-9] * n_qubits)
"""
The wait time before starting the readout acquisition (as of now: common to all
qubits).
"""

classification_refs = Array(
    name="references", value=[[0, 0.1]] * n_qubits, dtype=complex
)
"""
The classification reference points for classification of IQ data.
"""

cr_gate_amplitude = Array("cr_gate_amplitude", value=[0.2] * n_mq, dtype=float)
cr_gate_duration = Scalar("cr_gate_duration", value=100e-9, dtype=float)
"""
The parameters for the CR gates
"""

####################################################################################
#
# Add the X gate
#
####################################################################################

x_pulse = RFWaveform(
    xy_pulse_durations,
    GaussianEnvelope(),
    x180_pulse_amplitudes,
    xy_pulse_frequencies,
)
calibration_set.add_sq_gate("x", GATES.x, x_pulse, qubits, xy_awg)

####################################################################################
#
# Add the identity gate
#
####################################################################################

id_pulse = qcs.Delay(0)
calibration_set.add_sq_gate("id", GATES.id, id_pulse, qubits, xy_awg)

####################################################################################
#
# Add the X90 gate
#
####################################################################################

x90_pulse = RFWaveform(
    xy_pulse_durations,
    GaussianEnvelope(),
    x180_pulse_amplitudes / 2,
    xy_pulse_frequencies,
)
calibration_set.add_sq_gate("sx", GATES.x90, x90_pulse, qubits, xy_awg)

####################################################################################
#
# Add the RX gate
#
####################################################################################

# define the variable rotation angle (in radians)
rx_angle = Array("rx_angle", shape=n_qubits, dtype=float)

rx_pulse = RFWaveform(
    xy_pulse_durations,
    GaussianEnvelope(),
    rx_angle / np.pi * x180_pulse_amplitudes,
    xy_pulse_frequencies,
)
calibration_set.add_sq_gate(
    "rx",
    ParameterizedGate(PAULIS.rx, rx_angle),
    rx_pulse,
    qubits,
    xy_awg,
)

####################################################################################
#
# Add the Y gate
#
####################################################################################

y_pulse = RFWaveform(
    xy_pulse_durations,
    GaussianEnvelope(),
    x180_pulse_amplitudes,
    xy_pulse_frequencies,
    instantaneous_phase=np.pi / 2,
)
calibration_set.add_sq_gate("y", GATES.y, y_pulse, qubits, xy_awg)

####################################################################################
#
# Add the RY gate
#
####################################################################################

# define the variable rotation angle (in radians)
ry_angle = Array("ry_angle", shape=n_qubits, dtype=float)

ry_pulse = RFWaveform(
    xy_pulse_durations,
    GaussianEnvelope(),
    ry_angle / np.pi * x180_pulse_amplitudes,
    xy_pulse_frequencies,
    instantaneous_phase=np.pi / 2,
)
calibration_set.add_sq_gate(
    "ry",
    ParameterizedGate(PAULIS.ry, ry_angle),
    ry_pulse,
    qubits,
    xy_awg,
)
####################################################################################
#
# Add the Z gate
#
####################################################################################

calibration_set.add_sq_gate("z", GATES.z, PhaseIncrement(np.pi), qubits, xy_awg)

####################################################################################
#
# Add the Z90 gate
#
####################################################################################

calibration_set.add_sq_gate("z90", GATES.z90, PhaseIncrement(np.pi / 2), qubits, xy_awg)

####################################################################################
#
# Add the virtual Z gate (RZ)
#
####################################################################################

phi = Array(name="phi", shape=n_qubits, dtype=float)
calibration_set.add_sq_gate(
    "virtual_z",
    ParameterizedGate(PAULIS.rz, phi),
    PhaseIncrement(phi),
    qubits,
    xy_awg,
)

####################################################################################
#
# Add the CR gate
#
####################################################################################

if topology != "single_qubit":
    # define the instruction to compile
    zx = PAULIS.sigma_z & PAULIS.sigma_x
    cr_gate = ParametricGate([zx], ["beta"])

    # make it a base operation to create a linker
    angles = Array(name="beta", shape=(n_mq,), dtype=float)
    cr_param_gate = ParameterizedGate(cr_gate, angles)
    angle = angles * cr_gate_amplitude / np.pi

    if topology == "linear":
        stop = n_qubits if n_qubits % 2 == 0 else n_qubits - 1
        # extracting all the second labels of each pair = target qubits,
        # and create a new variable for the target qubits only
        xy_pulse_frequencies_target = Array(
            "xy_pulse_frequencies_target",
            dtype=float,
            value=xy_pulse_frequencies.value[1:stop:2],
        )
    elif topology == "custom":
        # extracting all the second label of each pair in given
        # connections = control qubits,
        # and create a new variable for the target qubits only
        labels = [label[1] for label in edge_list]
        sliced_frequencies = [xy_pulse_frequencies.value[i] for i in labels]
        xy_pulse_frequencies_target = Array(
            "xy_pulse_frequencies_target", dtype=float, value=sliced_frequencies
        )

    # pulse to link to that gate = drive on control qubit at
    # frequency of target qubit
    control_pulse = RFWaveform(
        cr_gate_duration, GaussianEnvelope(), angle, xy_pulse_frequencies_target
    )

    calibration_set.add_cr_gate(
        gate=cr_param_gate,
        qudits=pairs,
        sq_linker="sx",
        control_waveform=control_pulse,
        name="CR",
    )

####################################################################################
#
# Add the measurement
#
####################################################################################

# define the readout drive pulse and integration filter
readout_pulse = RFWaveform(
    readout_pulse_duration,
    SineEnvelope(),
    readout_pulse_amplitudes,
    readout_frequencies,
    instantaneous_phase=readout_pulse_phases,
)

# Define the integration filter for acquisition
integration_filter = RFWaveform(
    acquisition_duration, ConstantEnvelope(), 1, readout_frequencies
)

calibration_set.add_measurement(
    readout_pulse,
    qubits,
    readout_awg,
    dig,
    integration_filter,
    pre_delay=readout_pulse_delay,
    acquisition_delay=acquisition_delay,
    classification_references=classification_refs,
    name="measurement",
)


qcs.save(calibration_set, "calibration_set.qcs")
print("Done - Cal file generated successfully")

Done - Cal file generated successfully
